# Longor M2 launcher (RoboCasa OpenDrawer + GR00T)

Runs one frozen **M2** Longor episode on RoboCasa (`open_drawer`, `task_idx=0`) with GR00T N1.6 via the Longor HTTP adapter.

**Launcher contract:** `RUN_SCRIPT=examples/run_longor_task.py`

Orchestration and verdict logic live in Python scripts (`examples/run_longor_task.py`, optional `validation/run_robocasa_m2_gate.py`). This notebook only installs deps, starts servers, and invokes those scripts.

For the upstream-only **M1** primitive gate, use `colab/run_robocasa_primitive_gate.ipynb` instead.

## 1 — Clone Longor and mount Drive

Colab runtime: **Linux + NVIDIA GPU**. Set `GITHUB_TOKEN` (repo PAT) and `OPENAI_API_KEY` in Colab **Secrets** (optional `HF_TOKEN` for gated checkpoint download).

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive, userdata

github_token = userdata.get('GITHUB_TOKEN')
openai_api_key = userdata.get('OPENAI_API_KEY')

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

drive.mount('/content/drive')

RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ARTIFACT_DIR = Path(f'/content/drive/MyDrive/Colab/longor_m2_task/{RUN_UTC}')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['ARTIFACT_DIR'] = str(ARTIFACT_DIR)
os.environ['RUN_UTC'] = RUN_UTC
print('ARTIFACT_DIR=', ARTIFACT_DIR)
print('RUN_UTC=', RUN_UTC)

## 2 — Headless rendering + optional Hugging Face auth

In [ ]:
import os

os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
except Exception:
    pass

## 3 — Isaac-GR00T RoboCasa env + Longor installs

Minimal n1d6 RoboCasa stack (same `setup_RoboCasa.sh` patch as the M1 gate notebook). Longor runs in the RoboCasa sim venv; the GR00T HTTP adapter uses the Isaac-GR00T project venv.

In [ ]:
%%bash
set -euo pipefail
SETUP_LOG="/tmp/longor_m2_setup_${RUN_UTC:-manual}.log"
exec > >(tee "${SETUP_LOG}") 2>&1
trap 'cp -f "${SETUP_LOG}" "${ARTIFACT_DIR}/setup.log" 2>/dev/null || true' EXIT

cd /content/Longor
GROOT_ROOT=/content/Isaac-GR00T
GROOT_BRANCH=n1d6
MODEL_PATH=nvidia/GR00T-N1.6-3B
EMBODIMENT_TAG=ROBOCASA_PANDA_OMRON

apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  git git-lfs ffmpeg libaio-dev libegl1-mesa-dev libglu1-mesa python3.10-venv python3.10-dev
git lfs install

if [ ! -d "${GROOT_ROOT}/.git" ]; then
  git clone --branch "${GROOT_BRANCH}" --recurse-submodules \
    https://github.com/NVIDIA/Isaac-GR00T.git "${GROOT_ROOT}"
else
  git -C "${GROOT_ROOT}" fetch origin "${GROOT_BRANCH}"
  git -C "${GROOT_ROOT}" checkout "${GROOT_BRANCH}"
  git -C "${GROOT_ROOT}" submodule update --init --recursive
fi

cd "${GROOT_ROOT}"
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"

python3 - <<'PY'
from pathlib import Path
script = Path('gr00t/eval/sim/robocasa/setup_RoboCasa.sh')
text = script.read_text(encoding='utf-8')
needle = 'uv pip install '
patched = 'uv pip install --python "$UV_ENV/.venv/bin/python" '
if patched not in text:
    script.write_text(text.replace(needle, patched), encoding='utf-8')
PY

export UV_HTTP_TIMEOUT=120
for attempt in 1 2 3 4; do
  if uv sync; then
    break
  fi
  if [ "${attempt}" -eq 4 ]; then
    echo "uv sync failed after ${attempt} attempts" >&2
    exit 1
  fi
  sleep $((attempt * 15))
done
source .venv/bin/activate
bash gr00t/eval/sim/robocasa/setup_RoboCasa.sh

ROBO_PY="${GROOT_ROOT}/gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python"
GROOT_PY="${GROOT_ROOT}/.venv/bin/python"

cd /content/Longor
"${ROBO_PY}" -m ensurepip --upgrade || true
"${ROBO_PY}" -m pip install -U pip
"${ROBO_PY}" -m pip install -r deps/requirements.txt
"${ROBO_PY}" -m pip install -e .

uv pip install --python "${GROOT_PY}" -r deps/requirements.txt
uv pip install --python "${GROOT_PY}" -e .

echo "GROOT_ROOT=${GROOT_ROOT}" > /content/Longor/.colab_m2_env
echo "ROBO_PY=${ROBO_PY}" >> /content/Longor/.colab_m2_env
echo "GROOT_PY=${GROOT_PY}" >> /content/Longor/.colab_m2_env
echo "MODEL_PATH=${MODEL_PATH}" >> /content/Longor/.colab_m2_env
echo "EMBODIMENT_TAG=${EMBODIMENT_TAG}" >> /content/Longor/.colab_m2_env

"${ROBO_PY}" - <<'PY'
import gymnasium as gym
import robocasa  # noqa: F401
from robocasa.utils.gym_utils import GrootRoboCasaEnv  # noqa: F401
import robosuite  # noqa: F401
from envs.robocasa_env import OPEN_DRAWER_GYM_ENV_ID

try:
    gym.spec(OPEN_DRAWER_GYM_ENV_ID)
except Exception as exc:
    namespaces = sorted(
        {
            spec.namespace
            for spec in gym.envs.registry.values()
            if getattr(spec, "namespace", None)
        }
    )
    print(f"missing gym spec: {OPEN_DRAWER_GYM_ENV_ID}")
    print(f"registered namespaces (sample): {namespaces[:20]}")
    raise RuntimeError("RoboCasa PandaOmron gym namespace is not registered") from exc

print(f"RoboCasa + Longor import OK; gym spec ready: {OPEN_DRAWER_GYM_ENV_ID}")
PY

## 4 — Configure `.env` for M2 (RoboCasa + GR00T)

In [ ]:
from pathlib import Path

Path('.env').write_text(
    f'OPENAI_API_KEY={openai_api_key}\n'
    'VLA_POLICY=groot\n'
    'LOG_LEVEL=DEBUG\n'
    'LLM_MODEL=gpt-4.1-mini\n'
    'SAVE_VIDEO=1\n'
    f'VIDEO_PATH={ARTIFACT_DIR / "run.mp4"}\n',
    encoding='utf-8',
)
print('.env written (SIM_ENV=robocasa is default in src/config.py)')

## 5 — Start upstream GR00T (:5555) and Longor adapter (:8765)

In [ ]:
%%bash
set -euo pipefail
source /content/Longor/.colab_m2_env
cd /content/Longor

UPSTREAM_LOG="${ARTIFACT_DIR}/upstream_gr00t_server.log"
ADAPTER_LOG="${ARTIFACT_DIR}/longor_groot_adapter.log"

pkill -f run_gr00t_server.py || true
pkill -f services/groot_service.py || true
pkill -f services/run_vla_server.py || true
sleep 2

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export PYTHONUNBUFFERED=1

cd "${GROOT_ROOT}"
source .venv/bin/activate
nohup uv run python gr00t/eval/run_gr00t_server.py \
  --model-path "${MODEL_PATH}" \
  --embodiment-tag "${EMBODIMENT_TAG}" \
  --use-sim-policy-wrapper >"${UPSTREAM_LOG}" 2>&1 &

cd /content/Longor
uv pip install --python "${GROOT_PY}" fastapi "uvicorn[standard]" python-multipart pillow
for _ in $(seq 1 180); do
  if curl -fsS http://127.0.0.1:5555 >/dev/null 2>&1 || grep -q 'listening' "${UPSTREAM_LOG}" 2>/dev/null; then
    break
  fi
  sleep 5
done

# 8 actions/call: matches M1; RoboCasa frames are valid only every 4th step
export VLA_ACTIONS_PER_CHUNK=8
nohup "${GROOT_PY}" services/run_vla_server.py >"${ADAPTER_LOG}" 2>&1 &

for _ in $(seq 1 120); do
  HEALTH_JSON="$(curl -fsS http://127.0.0.1:8765/health 2>/dev/null || true)"
  if [ -n "${HEALTH_JSON}" ]; then
    echo "${HEALTH_JSON}"
    echo
    echo 'Longor GR00T adapter healthy'
    exit 0
  fi
  sleep 5
done

echo 'adapter failed to become healthy' >&2
tail -50 "${ADAPTER_LOG}" >&2
exit 1

## 6 — Run one Longor M2 episode

Frozen M2 slice: `--task open_drawer --task-idx 0`.

In [ ]:
%%bash
set -uo pipefail
source /content/Longor/.colab_m2_env
cd /content/Longor

export VIDEO_PATH="${ARTIFACT_DIR}/run.mp4"
TASK_NAME=open_drawer
TASK_IDX=0
RUN_LOG="${ARTIFACT_DIR}/run_longor_task.log"

set +e
"${ROBO_PY}" examples/run_longor_task.py \
  --task "$TASK_NAME" \
  --task-idx "$TASK_IDX" 2>&1 | tee "$RUN_LOG"
LONGOR_EXIT=${PIPESTATUS[0]}
set -e

echo "LONGOR_EXIT_CODE=$LONGOR_EXIT"
echo "RUN_LOG=$RUN_LOG"
grep -E "^PASS task=|^FAIL task=" "$RUN_LOG" | tail -n 1 || true
exit ${LONGOR_EXIT}

## 7 — Optional M2 gate (live stages A + B)

Invokes `validation/run_robocasa_m2_gate.py --run-live-stages` (Stage B calls `examples/run_longor_task.py`). Skip if you only need the single-episode run above.

In [ ]:
%%bash
set -euo pipefail
source /content/Longor/.colab_m2_env
cd /content/Longor

M2_OUT="${ARTIFACT_DIR}/m2_gate"
"${ROBO_PY}" validation/run_robocasa_m2_gate.py \
  --output-root "$M2_OUT" \
  --run-utc "$RUN_UTC" \
  --run-live-stages 2>&1 | tee "${ARTIFACT_DIR}/m2_gate_launcher.log"
echo "m2_gate_dir=${M2_OUT}/${RUN_UTC}"

## 8 — Stop policy servers

In [ ]:
%%bash
set -euo pipefail
pkill -f run_gr00t_server.py || true
pkill -f services/groot_service.py || true
pkill -f services/run_vla_server.py || true
echo 'servers stopped'